# 속성 증류·파인튜닝 상한

중요: 아래 결과는 PA-100K 100장 proxy의 6개 field best observed result다. 프로젝트 운영 속성인 color, clothing, texture, quality, occlusion의 ceiling이 아니다. 고정 CLIP+속성 head와 부분 fine-tuning을 동일한 deterministic 5-fold로 비교하며, 실제 CCTV identity ceiling이나 Sonnet teacher distillation 결과로 해석하지 않는다. pattern 라벨은 manifest에서 모두 0건이라 제외했다.

In [1]:
import json
from pathlib import Path

from IPython.display import Markdown, display

root = Path.cwd().parent if Path.cwd().name == 'experiments' else Path.cwd()
short_run = json.loads((root / 'experiments/results/attribute-ceiling.json').read_text(encoding='utf-8'))
long_run = json.loads((root / 'experiments/results/attribute-ceiling-ft40.json').read_text(encoding='utf-8'))
assert short_run['status'] == 'valid' and long_run['status'] == 'valid'
assert short_run['examples'] == long_run['examples'] == 100
fields = list(short_run['fields'])
print({'examples': short_run['examples'], 'folds': short_run['folds'], 'device': short_run['device'], 'fields': fields})

{'examples': 100, 'folds': 5, 'device': 'cuda', 'fields': ['gender', 'age', 'viewpoint', 'accessory_present', 'sleeve', 'bottom_type']}


In [2]:
rows = [
    ('고정 CLIP + head (120 epoch)', short_run['linear_probe_validation']),
    ('부분 fine-tuning (8 epoch)', short_run['partial_clip_finetune_validation']),
    ('부분 fine-tuning (40 epoch)', long_run['partial_clip_finetune_validation']),
]
header = '| 방법 | 속성 정확도 | Macro-F1 |'
divider = '|---|---:|---:|'
body = '\n'.join(f'| {name} | {metrics["attribute_accuracy"]:.2%} | {metrics["macro_f1"]:.2%} |' for name, metrics in rows)
display(Markdown('\n'.join([header, divider, body])))

| 방법 | 속성 정확도 | Macro-F1 |
|---|---:|---:|
| 고정 CLIP + head (120 epoch) | 71.67% | 41.01% |
| 부분 fine-tuning (8 epoch) | 68.83% | 30.09% |
| 부분 fine-tuning (40 epoch) | 72.67% | 43.55% |

In [3]:
field_rows = '\n'.join(
    f'| {field} | {short_run["linear_probe_validation"]["field_accuracy"][i]:.2%} | {long_run["partial_clip_finetune_validation"]["field_accuracy"][i]:.2%} |'
    for i, field in enumerate(fields)
)
display(Markdown('\n'.join([
    '| 속성 | 고정 CLIP + head | fine-tuning 40 epoch |',
    '|---|---:|---:|',
    field_rows,
])))

| 속성 | 고정 CLIP + head | fine-tuning 40 epoch |
|---|---:|---:|
| gender | 66.00% | 69.00% |
| age | 97.00% | 97.00% |
| viewpoint | 41.00% | 41.00% |
| accessory_present | 62.00% | 64.00% |
| sleeve | 83.00% | 84.00% |
| bottom_type | 81.00% | 81.00% |

## 해석

현재 proxy의 최선은 평균 6-field 속성 정확도 약 72.67%다. age와 bottom_type은 높지만 viewpoint와 gender가 병목이며, 데이터에 색상·질감·신원 ID·영상 track 라벨이 없어 해당 속성의 배포 상한은 측정되지 않았다. Sonnet 또는 Qwen의 실제 response-level teacher-label distillation은 승인된 teacher 응답이 생긴 뒤 같은 track-held-out 평가로 별도 측정해야 한다.